In [1]:
# Init
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv(override=True)

api_key = os.getenv("API_KEY")
base_url = os.getenv("BASE_URL")
model = os.getenv("MODEL")

client = Anthropic(api_key=api_key, base_url=base_url)

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

schema = {
    "type": "object",
    "properties": {
        "tasks": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "task": {"type": "string", "description": "Description of task"},
                },
                "required": ["task"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["tasks"],
    "additionalProperties": False,
}

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 5000,
        "messages": messages,
        "output_config": {"format": {"type": "json_schema", "schema": schema}},
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    #return message.content[0].text
    #
    # a fix for ThinkingBlock error
    return "".join(b.text for b in message.content if b.type == "text")

In [2]:
import json

def generate_dataset():
  prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

  messages = []
  add_user_message(messages, prompt)
  text = chat(messages)
  return json.loads(text)


generate_dataset()

{'tasks': [{'task': 'Write a single Python function using boto3 that lists the names of all S3 buckets in an AWS account whose creation date is older than 365 days.'},
  {'task': "Write a single JSON object representing an AWS IAM policy that allows read-only access (s3:GetObject and s3:ListBucket) to only the bucket named 'reports-archive'."},
  {'task': "Write a single regex that matches valid AWS EC2 instance IDs (e.g., i-0abcd1234efgh5678), which start with 'i-' followed by 8 or 17 hexadecimal characters."}]}

Example output:

```json
{
    'tasks': [
        {
            'task': 'Write a single Python function using boto3 that lists all S3 buckets in an AWS account and returns only the names of buckets whose creation date is before a given date.'
        },
        {
            'task': "Write a single JSON IAM policy document that allows read-only access (s3:GetObject and s3:ListBucket) to one specific S3 bucket named 'reports-archive' and denies all delete actions on that bucket."},
        {
            'task': "Write a single regex that matches valid AWS EC2 instance IDs (format 'i-' followed by 8 or 17 lowercase hexadecimal characters)."
        }
    ]
}
```


In [3]:
dataset = generate_dataset()

# Save dataset to file
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [4]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [5]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

print(dataset)

result = run_eval(dataset["tasks"])

print(json.dumps(result, indent=2))

{'tasks': [{'task': 'Write a Python function using boto3 that takes an S3 bucket name and returns the total number of objects in the bucket, handling pagination.'}, {'task': "Write a JSON IAM policy document that allows read-only access (s3:GetObject and s3:ListBucket) to a single S3 bucket named 'app-logs-prod'."}, {'task': "Write a regex that matches valid AWS EC2 instance IDs (for example i-0abcd1234efgh5678), which start with 'i-' followed by 8 or 17 hexadecimal characters."}]}
[
  {
    "output": "{\"tasks\":[{\"task\":\"Python function using boto3 that takes an S3 bucket name and returns the total number of objects in the bucket, handling pagination:\\n\\nimport boto3\\n\\ndef count_s3_objects(bucket_name):\\n    s3 = boto3.client('s3')\\n    paginator = s3.get_paginator('list_objects_v2')\\n    total = 0\\n    for page in paginator.paginate(Bucket=bucket_name):\\n        total += page.get('KeyCount', 0)\\n    return total\\n\\nExplanation: The function creates an S3 client and u